In [2]:
import json
from typing import TypedDict, Annotated, Literal
import operator
from pydantic import BaseModel, Field
from langchain_groq import  ChatGroq 

from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

In [3]:
# 1. DETERMINISTIC MATH / CIRCUIT BREAKER
# ==========================================
def factorial(n: int) -> int:
    result = 1
    for i in range(2, n + 1): result *= i
    return result

def combinations(n: int, k: int) -> int:
    if k < 0 or k > n: return 0
    return factorial(n) // (factorial(k) * factorial(n - k))

def binomial_probability(n: int, k: int, p: float) -> float:
    return combinations(n, k) * (p ** k) * ((1 - p) ** (n - k))

# ==========================================
# 2. STATE AND DATA MODELS
# ==========================================
class AgentDecision(BaseModel):
    intervention: str = Field(description="Recommended medical intervention")
    risk_level: str = Field(description="HIGH, MEDIUM, LOW")
    extracted_anomalies: int

class ClinicalState(TypedDict):
    patient_data: str
    decision: AgentDecision
    audit_passed: bool
    audit_reason: str
    human_feedback: str

In [4]:
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0,
    # groq_api_key="your-api-key" # Optional if GROQ_API_KEY environment variable is set
)